In [25]:
import scipy
from scipy.ndimage import binary_dilation
import numpy as np
from PIL import Image
from skimage.morphology import skeletonize
import sknw
import rasterio

def neighbours(x, y, image):
    i = image
    x1, y1, x_1, y_1 = x+1, y-1, x-1, y+1
    return [i[y1][x],  i[y1][x1],   i[y][x1],  i[y_1][x1],  
            i[y_1][x], i[y_1][x_1], i[y][x_1], i[y1][x_1]]

def transitions(neighbours):
    n = neighbours + neighbours[0:1]
    return sum((n1, n2) == (0, 1) for n1, n2 in zip(n, n[1:]))

def zhang_suen_thinning(image):
    changing1 = changing2 = [(-1, -1)]
    while changing1 or changing2:
        changing1 = []
        for y in range(1, len(image) - 1):
            for x in range(1, len(image[0]) - 1):
                P2,P3,P4,P5,P6,P7,P8,P9 = n = neighbours(x, y, image)
                if (image[y][x] == 1 and P4 * P6 * P8 == 0 and 
                    P2 * P4 * P6 == 0 and transitions(n) == 1 and 
                    2 <= sum(n) <= 6):
                    changing1.append((x,y))
        for x, y in changing1: image[y][x] = 0
        
        changing2 = []
        for y in range(1, len(image) - 1):
            for x in range(1, len(image[0]) - 1):
                P2,P3,P4,P5,P6,P7,P8,P9 = n = neighbours(x, y, image)
                if (image[y][x] == 1 and P2 * P6 * P8 == 0 and 
                    P2 * P4 * P8 == 0 and transitions(n) == 1 and 
                    2 <= sum(n) <= 6):
                    changing2.append((x,y))
        for x, y in changing2: image[y][x] = 0
        
    return image

def convert_tif_to_bw_png(input_tif):
    # Open the raster file
    with rasterio.open(input_tif) as dataset:
        image_array = dataset.read(1)

    image_array = (image_array - np.min(image_array)) / (np.max(image_array) - np.min(image_array)) * 255
    image_array = image_array.astype(np.uint8)  # Convert to 8bit

    # Convert to b&w
    bw_image = Image.fromarray(image_array).convert('L').point(lambda x: 0 if x < 128 else 255, '1')
    return bw_image

input_tif = "/Users/sawyerj/Downloads/srccopy/pred_mask_patch9-8.tif"
bw_image = convert_tif_to_bw_png(input_tif)

bw_image = bw_image.convert("L")
threshold = 128
binary_image = bw_image.point(lambda p: 0 if p < threshold else 255)

numpy_array = np.array(binary_image)

# Ensure 0/1 values
numpy_array = numpy_array // 255 

# dilation!
struct = np.ones((3, 3), dtype=bool)
dilated_array = binary_dilation(numpy_array, structure=struct).astype(np.uint8)
dilated_image = Image.fromarray(dilated_array * 255)


# Apply zhang-Suen thinning
dilated_array = zhang_suen_thinning(dilated_array.tolist())
thinned_array = np.array(dilated_array)
thinned_array = thinned_array.astype(np.uint8) 
thinned_array = thinned_array[1:-1, 1:-1]

# Skeletonization
ske = skeletonize(thinned_array).astype(np.uint8)

# Build graph from skeleton
graph = sknw.build_sknw(ske)

# Create a new image to overlay the skeleton on the binary image
skeleton_image = Image.fromarray(thinned_array * 255)

# Overlay skeleton on top of the binary image (black lines)
for (s, e) in graph.edges():
    ps = graph[s][e]['pts']
    for point in ps:
        skeleton_image.putpixel((point[1], point[0]), 255)  # (x, y) coordinates, using 0 for black

# Show the image with the skeleton
skeleton_image.show()